In [4]:
import sys
import time
import random
from hypercomplex_engine import FastStandard, FastSplit
fs, fp = FastStandard(), FastSplit()
sys.setrecursionlimit(100000)

def conjsign(i): return 1 if i == 0 else -1
def ref(i, j, n, top_split):
    """Independent monomial Cayley-Dickson product e_i*e_j in 2^n dims -> sign (index is i^j).
    Convention (a,b)(c,d) = (ac + g d*b, da + b c*), g=-1 standard; top doubling g=+1 if top_split.
    Iterative from top level down (no recursion), tracking accumulated sign."""
    sign = 1
    for lvl in range(n, 0, -1):
        h = 1 << (lvl - 1)
        g = (+1 if (top_split and lvl == n) else -1)
        iu, ju = i >= h, j >= h
        if iu: i -= h
        if ju: j -= h
        if not iu and not ju:   # (a,0)(c,0) -> ac
            pass
        elif not iu and ju:     # (a,0)(0,d) -> (0, d a): swap operand order
            i, j = j, i
        elif iu and not ju:     # (0,b)(c,0) -> (0, b c*)
            sign *= conjsign(j)
        else:                   # (0,b)(0,d) -> (g d* b, 0)
            sign *= g * conjsign(j)
            i, j = j, i
    return sign   # level-0 product is 1*1

# --- sanity: ref agrees with previous exhaustive recursion results
print("exhaustive, all pairs, standard & split (top doubling):")
t0=time.time()
for n in range(1, 10):
    N = 1 << n; bs = bp = 0
    for i in range(N):
        for j in range(N):
            if fs.multiply_indices(i, j) != (ref(i, j, n, False), i ^ j): bs += 1
            if fp.multiply_indices(i, j, n) != (ref(i, j, n, True), i ^ j): bp += 1
    print(f"  n={n} ({N:3d}-dim): standard mismatches {bs}, split mismatches {bp}")
print(f"  ({time.time()-t0:.1f}s)")

print("\nrandom pairs at huge dimension (structured + uniform):")
random.seed(1)
def rnd_idx(n):
    mode = random.random()
    if mode < .4: return random.getrandbits(n)
    if mode < .7: return (1 << random.randrange(n)) | (random.getrandbits(n) if random.random()<.3 else 0)  # powers of two (+noise)
    if mode < .85: return random.getrandbits(n) & ~((1 << random.randrange(n)) - 1)                        # many trailing zeros
    return (1 << random.randrange(n)) - 1                                                                   # all-ones masks
for n in (16, 64, 256, 1024, 4096):
    bs = bp = 0; K = 20000 if n<=1024 else 4000
    for _ in range(K):
        i, j = rnd_idx(n), rnd_idx(n)
        if fs.multiply_indices(i, j) != (ref(i, j, n, False), i ^ j): bs += 1
        if fp.multiply_indices(i, j, n) != (ref(i, j, n, True), i ^ j): bp += 1
    print(f"  n={n:5d}: {K} pairs  standard mismatches {bs}, split mismatches {bp}")

exhaustive, all pairs, standard & split (top doubling):
  n=1 (  2-dim): standard mismatches 0, split mismatches 0
  n=2 (  4-dim): standard mismatches 0, split mismatches 0
  n=3 (  8-dim): standard mismatches 0, split mismatches 0
  n=4 ( 16-dim): standard mismatches 0, split mismatches 0
  n=5 ( 32-dim): standard mismatches 0, split mismatches 0
  n=6 ( 64-dim): standard mismatches 0, split mismatches 0
  n=7 (128-dim): standard mismatches 0, split mismatches 0
  n=8 (256-dim): standard mismatches 0, split mismatches 0
  n=9 (512-dim): standard mismatches 0, split mismatches 0
  (1.5s)

random pairs at huge dimension (structured + uniform):
  n=   16: 20000 pairs  standard mismatches 0, split mismatches 0
  n=   64: 20000 pairs  standard mismatches 0, split mismatches 0
  n=  256: 20000 pairs  standard mismatches 0, split mismatches 0
  n= 1024: 20000 pairs  standard mismatches 0, split mismatches 0
  n= 4096: 4000 pairs  standard mismatches 0, split mismatches 0
